[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/03_advanced_architectures/gnn.ipynb)


# 14 . Graph Neural Networks from Scratch

## Concept — Graph Neural Networks (revision notes)

**What it is**
- A neural network that operates on graphs: a set of nodes (each with a
  feature vector) and edges that say which nodes are related.
- Core operation is *message passing*: every node gathers ("aggregates")
  information from its neighbours, combines it with its own state, and
  updates its representation. Stacking `k` layers lets information travel
  `k` hops.

**Why it matters**
- Molecules, social networks, citation graphs, road networks, and
  recommender systems are graphs, not grids or sequences. A GNN shares
  weights across all nodes and handles any graph size or node ordering.

**When to use**
- The relationships between examples carry signal (e.g. papers that cite
  each other tend to share a topic: *homophily*). If the graph is
  uninformative, a plain MLP on node features may do just as well.

**Math & intuition**
- Message passing: `h_i' = UPDATE(h_i, AGG_{j in N(i)} MESSAGE(h_j))`.
- GCN layer (Kipf & Welling): `H' = act( A_hat H W )` with
  `A_hat = D^-1/2 (A + I) D^-1/2`. Self-loops (`+ I`) keep a node's own
  features; the symmetric degree normalisation stops high-degree nodes from
  dominating.
- GAT-style attention: instead of fixed weights from degrees, learn
  `alpha_ij = softmax_j( LeakyReLU( a^T [W h_i || W h_j] ) )` over the
  neighbours `j` of `i`, then `h_i' = act( sum_j alpha_ij W h_j )`.
- Permutation equivariance: relabelling the nodes just permutes the
  output rows in the same way. Too many layers cause *over-smoothing*:
  node representations converge to each other.
- We implement everything in pure PyTorch (no `torch_geometric`).

### 1. A synthetic graph: stochastic block model with noisy features

**What this cell does (plain language):** we build a graph with 3 hidden communities of 40 nodes each. Nodes in the same community are linked with probability 0.12 and nodes in different communities with probability 0.01 (so the graph is *homophilous*). Node features are only weakly informative: a class-dependent mean buried in a lot of noise. Only 5 labelled nodes per class are used for training, which is the classic semi-supervised setting where the graph structure should help.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(1)   # tiny models: avoids thread-oversubscription slowdowns
torch.manual_seed(0)

n_classes, n_per, F_in = 3, 40, 16
N = n_classes * n_per
y = torch.arange(n_classes).repeat_interleave(n_per)

# Edge probabilities: dense inside a community, sparse across communities
same = (y[:, None] == y[None, :]).float()
P = 0.12 * same + 0.01 * (1 - same)
A_rand = (torch.rand(N, N) < P).float()
A = torch.triu(A_rand, diagonal=1)
A = A + A.T                                    # undirected, no self loops
edge_index = A.nonzero().T                     # shape (2, num_directed_edges): row 0 = source, row 1 = target

# Weakly informative features: class mean (small) + big noise
class_means = torch.randn(n_classes, F_in)
X = 0.6 * class_means[y] + torch.randn(N, F_in)

# Semi-supervised split: 5 train labels per class, 10 val per class, the rest test
train_mask = torch.zeros(N, dtype=torch.bool); val_mask = train_mask.clone(); test_mask = train_mask.clone()
for c in range(n_classes):
    idx = (y == c).nonzero().flatten()
    idx = idx[torch.randperm(len(idx))]
    train_mask[idx[:5]] = True; val_mask[idx[5:15]] = True; test_mask[idx[15:]] = True

deg = A.sum(1)
print("nodes:", N, "| directed edges:", edge_index.shape[1], "| mean degree:", round(deg.mean().item(), 2))
print("isolated nodes:", int((deg == 0).sum()))
frac_same = (y[edge_index[0]] == y[edge_index[1]]).float().mean().item()
print("fraction of edges joining same-class nodes (homophily):", round(frac_same, 3))
print("train/val/test sizes:", int(train_mask.sum()), int(val_mask.sum()), int(test_mask.sum()))

nodes: 120 | directed edges: 682 | mean degree: 5.68
isolated nodes: 2
fraction of edges joining same-class nodes (homophily): 0.824
train/val/test sizes: 15 30 75


### 2. Adjacency, self-loops and symmetric normalisation

**What this cell does (plain language):** we turn the raw adjacency matrix `A` into the GCN propagation matrix `A_hat = D^-1/2 (A + I) D^-1/2`. Adding the identity gives every node a self-loop (so a node keeps its own features), and dividing by square-rooted degrees makes the weight between `i` and `j` equal `1/sqrt(d_i d_j)`. We inspect the result: it stays symmetric, and each row no longer sums to exactly 1 (that would be random-walk normalisation).

In [2]:
def normalize_adj(A):
    A_tilde = A + torch.eye(A.shape[0])            # add self-loops
    d = A_tilde.sum(1)                              # degree including the self-loop (>= 1, so no division by zero)
    d_inv_sqrt = d.pow(-0.5)
    return d_inv_sqrt[:, None] * A_tilde * d_inv_sqrt[None, :]

A_hat = normalize_adj(A)
print("symmetric:", torch.allclose(A_hat, A_hat.T))
print("row sums (first 5):", [round(v, 3) for v in A_hat.sum(1)[:5].tolist()])
print("weight between node 0 and itself:", round(A_hat[0, 0].item(), 4), "= 1/(deg0+1) =", round(1 / (deg[0].item() + 1), 4))

symmetric: True
row sums (first 5): [0.91, 1.135, 0.954, 0.901, 0.795]
weight between node 0 and itself: 0.1429 = 1/(deg0+1) = 0.1429


### 3. Message passing by hand: gather, aggregate, update

**What this cell does (plain language):** before any GCN formula, we implement the plain-language idea with an edge list. For every directed edge `(src -> dst)` the message is simply the source node's feature vector; `index_add_` sums the messages that arrive at each destination (the aggregation); we then divide by the in-degree to take a MEAN. We verify this equals the dense matrix computation `D^-1 A X`. This gather / scatter pattern is exactly how libraries like torch_geometric implement sparse layers.

In [3]:
def mean_aggregate(x, edge_index):
    src, dst = edge_index
    messages = x[src]                                   # gather: one row per edge (E, F)
    out = torch.zeros_like(x)
    out.index_add_(0, dst, messages)                    # scatter-add messages onto their destination node
    count = torch.zeros(x.shape[0]).index_add_(0, dst, torch.ones(len(dst)))
    return out / count.clamp(min=1)[:, None]            # mean (clamp avoids 0/0 for isolated nodes)

agg = mean_aggregate(X, edge_index)
dense = (A / A.sum(1).clamp(min=1)[:, None]) @ X        # D^-1 A X
print("edge-list aggregation == dense D^-1 A X:", torch.allclose(agg, dense, atol=1e-5))

# Intuition: neighbourhood averaging denoises features. Compare class separability before/after.
def class_mean_gap(h):
    mu = torch.stack([h[y == c].mean(0) for c in range(n_classes)])
    within = torch.stack([h[y == c].var(0).mean() for c in range(n_classes)]).mean()
    between = mu.var(0).mean()
    return (between / within).item()
print("between/within class variance ratio, raw features :", round(class_mean_gap(X), 3))
print("between/within class variance ratio, after 1 hop  :", round(class_mean_gap(agg), 3))
print("between/within class variance ratio, after 2 hops :", round(class_mean_gap(mean_aggregate(agg, edge_index)), 3))

edge-list aggregation == dense D^-1 A X: True
between/within class variance ratio, raw features : 0.355
between/within class variance ratio, after 1 hop  : 0.831
between/within class variance ratio, after 2 hops : 2.16


### 4. A GCN layer with a dense adjacency

**What this cell does (plain language):** `H' = A_hat H W + b` as an `nn.Module`. The adjacency is precomputed once and passed to `forward`. The order `A_hat @ (H @ W)` is chosen on purpose: multiplying features by `W` first shrinks the feature dimension before the expensive N x N product when the output is narrower than the input. Dense adjacency costs O(N^2) memory, fine for 120 nodes but impossible for large graphs.

In [4]:
class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))
    def forward(self, x, A_hat):
        return A_hat @ self.lin(x) + self.bias       # propagate AFTER the linear transform

torch.manual_seed(0)
layer = DenseGCNLayer(F_in, 8)
out = layer(X, A_hat)
print("output shape:", tuple(out.shape))
# parameter count does not depend on N, the number of nodes
print("#params:", sum(p.numel() for p in layer.parameters()))

output shape: (120, 8)
#params: 136


### 5. The same layer with a sparse adjacency

**What this cell does (plain language):** real graphs are sparse: here only a few percent of the N x N entries are non-zero. We store `A_hat` as a `torch.sparse_coo_tensor` (built from the edge list plus self-loops, with normalisation weights computed from degrees) and use `torch.sparse.mm`. We copy the weights of the dense layer into the sparse one and check the outputs match, then compare how many numbers each representation stores.

In [5]:
def sparse_norm_adj(edge_index, N):
    loops = torch.arange(N)
    row = torch.cat([edge_index[0], loops]); col = torch.cat([edge_index[1], loops])
    deg = torch.zeros(N).index_add_(0, row, torch.ones(len(row)))        # degree incl. self-loop
    w = deg[row].pow(-0.5) * deg[col].pow(-0.5)                          # 1/sqrt(d_i d_j) per edge
    return torch.sparse_coo_tensor(torch.stack([row, col]), w, (N, N), check_invariants=False).coalesce()

class SparseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))
    def forward(self, x, A_sp):
        return torch.sparse.mm(A_sp, self.lin(x)) + self.bias

A_sp = sparse_norm_adj(edge_index, N)
sp_layer = SparseGCNLayer(F_in, 8)
sp_layer.load_state_dict(layer.state_dict())            # same weights as the dense layer
out_sp = sp_layer(X, A_sp)
print("sparse == dense output:", torch.allclose(out_sp, out, atol=1e-5))
print("dense stores", A_hat.numel(), "numbers | sparse stores", A_sp._nnz(), "non-zeros (", round(100 * A_sp._nnz() / A_hat.numel(), 1), "% )")

# gradients also work through the sparse matmul
sp_layer(X, A_sp).sum().backward()
print("grad on lin.weight exists:", sp_layer.lin.weight.grad is not None)

sparse == dense output: True
dense stores 14400 numbers | sparse stores 802 non-zeros ( 5.6 % )
grad on lin.weight exists: True


### 6. A 2-layer GCN for node classification

**What this cell does (plain language):** we stack two GCN layers (`F_in -> 16 -> 3`) with ReLU and dropout in between, and train with cross-entropy computed ONLY on the 15 labelled training nodes. All 120 nodes still take part in the forward pass, so unlabelled nodes influence the result through message passing; that is what 'semi-supervised' means here. We report train, validation, and test accuracy, and keep the weights of the epoch with the best validation accuracy.

In [6]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden, n_cls, p=0.5):
        super().__init__()
        self.l1, self.l2, self.p = DenseGCNLayer(in_dim, hidden), DenseGCNLayer(hidden, n_cls), p
    def forward(self, x, A_hat):
        x = F.dropout(x, self.p, self.training)
        x = F.relu(self.l1(x, A_hat))
        x = F.dropout(x, self.p, self.training)
        return self.l2(x, A_hat)                 # raw logits (no softmax: cross_entropy applies it)

def accuracy(logits, mask):
    return (logits[mask].argmax(1) == y[mask]).float().mean().item()

def fit(model, forward, epochs=150, lr=0.02, wd=5e-4, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_state = -1, None
    for ep in range(epochs):
        model.train()
        logits = forward()
        loss = F.cross_entropy(logits[train_mask], y[train_mask])   # loss on labelled nodes only
        opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            logits = forward()
        val = accuracy(logits, val_mask)
        if val > best_val:
            best_val, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
        if verbose and ep % 50 == 0:
            print(f"epoch {ep:3d} | loss {loss.item():.3f} | train {accuracy(logits, train_mask):.2f} | val {val:.2f}")
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        logits = forward()
    return accuracy(logits, train_mask), accuracy(logits, val_mask), accuracy(logits, test_mask)

torch.manual_seed(0)
gcn = GCN(F_in, 16, n_classes)
tr, va, te = fit(gcn, lambda: gcn(X, A_hat))
print(f"GCN  | train {tr:.2f} | val {va:.2f} | test {te:.2f}")

epoch   0 | loss 1.093 | train 0.40 | val 0.23
epoch  50 | loss 0.011 | train 1.00 | val 1.00


epoch 100 | loss 0.039 | train 1.00 | val 1.00
GCN  | train 1.00 | val 1.00 | test 0.93


### 7. Baseline: an MLP that ignores the graph

**What this cell does (plain language):** to see whether the graph is actually helping, we train an MLP of similar size on the same features and the same 15 labels. It treats every node independently. Because the node features are noisy and only 15 labels are available, we expect it to do clearly worse than the GCN, which averages noise away over neighbours.

In [7]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, n_cls, p=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden, n_cls))
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
mlp = MLP(F_in, 16, n_classes)
tr_m, va_m, te_m = fit(mlp, lambda: mlp(X), verbose=False)
print(f"MLP  | train {tr_m:.2f} | val {va_m:.2f} | test {te_m:.2f}")
print(f"GCN  | train {tr:.2f} | val {va:.2f} | test {te:.2f}")
print("GCN beats MLP on test:", te > te_m)

MLP  | train 0.93 | val 0.83 | test 0.71
GCN  | train 1.00 | val 1.00 | test 0.93
GCN beats MLP on test: True


### 8. A GAT-style attention layer on the edge list

**What this cell does (plain language):** instead of fixed degree-based weights, each edge gets a LEARNED weight. For every edge `j -> i` we compute a score `e_ij = LeakyReLU(a_src . Wh_j + a_dst . Wh_i)`, then a softmax over all edges that arrive at node `i` (so the weights over a node's neighbourhood sum to 1), then sum the weighted messages. The per-destination softmax is the tricky bit; we implement it with `scatter_reduce` (max for numerical stability) and `index_add_` (sum). Self-loops are added so a node can attend to itself. This is a single attention head; real GATs concatenate several.

In [8]:
class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, slope=0.2):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Parameter(torch.randn(out_dim) * 0.1)   # attention vector, source half
        self.a_dst = nn.Parameter(torch.randn(out_dim) * 0.1)   # attention vector, destination half
        self.slope = slope
        self.last_alpha = None
    def forward(self, x, edge_index):
        N = x.shape[0]
        loops = torch.arange(N)
        src = torch.cat([edge_index[0], loops]); dst = torch.cat([edge_index[1], loops])   # add self-loops
        h = self.W(x)
        # a^T [h_i || h_j] decomposes into a_dst . h_i + a_src . h_j
        e = F.leaky_relu((h @ self.a_src)[src] + (h @ self.a_dst)[dst], self.slope)         # (E,)
        # softmax over incoming edges of each destination node
        e_max = torch.full((N,), -1e9).scatter_reduce(0, dst, e, reduce="amax", include_self=True)
        e_exp = torch.exp(e - e_max[dst])                                                    # subtract max for stability
        denom = torch.zeros(N).index_add_(0, dst, e_exp)
        alpha = e_exp / denom[dst]                                                           # (E,) attention weights
        self.last_alpha, self.last_edges = alpha.detach(), (src, dst)
        out = torch.zeros_like(h).index_add_(0, dst, alpha[:, None] * h[src])               # weighted sum of messages
        return out

torch.manual_seed(0)
gat_layer = GATLayer(F_in, 8)
out = gat_layer(X, edge_index)
alpha_sum = torch.zeros(N).index_add_(0, gat_layer.last_edges[1], gat_layer.last_alpha)
print("output shape:", tuple(out.shape))
print("attention weights over each node's neighbourhood sum to 1:", torch.allclose(alpha_sum, torch.ones(N), atol=1e-5))
print("(an isolated node can only attend to itself, so its single weight is 1.0)\nuntrained alpha range:", round(gat_layer.last_alpha.min().item(), 3), "-", round(gat_layer.last_alpha.max().item(), 3))

output shape: (120, 8)
attention weights over each node's neighbourhood sum to 1: True
(an isolated node can only attend to itself, so its single weight is 1.0)
untrained alpha range: 0.065 - 1.0


### 9. Training a 2-layer GAT and inspecting attention

**What this cell does (plain language):** we plug two `GATLayer`s into a classifier and train it exactly like the GCN. Afterwards we look at what attention learned: we separate the edges into same-class and different-class edges and compare the average attention weight each group receives. If the model discovered that same-class neighbours are more trustworthy, same-class edges should receive more weight than their share would suggest. We print what we see; with so few labels the effect may be small.

In [9]:
class GAT(nn.Module):
    def __init__(self, in_dim, hidden, n_cls, p=0.5):
        super().__init__()
        self.l1, self.l2, self.p = GATLayer(in_dim, hidden), GATLayer(hidden, n_cls), p
    def forward(self, x, edge_index):
        x = F.dropout(x, self.p, self.training)
        x = F.elu(self.l1(x, edge_index))
        x = F.dropout(x, self.p, self.training)
        return self.l2(x, edge_index)

torch.manual_seed(0)
gat = GAT(F_in, 16, n_classes)
tr_g, va_g, te_g = fit(gat, lambda: gat(X, edge_index), verbose=False)
print(f"GAT  | train {tr_g:.2f} | val {va_g:.2f} | test {te_g:.2f}")

gat.eval()
with torch.no_grad():
    gat(X, edge_index)
alpha = gat.l1.last_alpha; src, dst = gat.l1.last_edges
not_loop = src != dst
same_cls = (y[src] == y[dst]) & not_loop
diff_cls = (y[src] != y[dst]) & not_loop
print("mean attention on same-class edges     :", round(alpha[same_cls].mean().item(), 4), f"({int(same_cls.sum())} edges)")
print("mean attention on different-class edges:", round(alpha[diff_cls].mean().item(), 4), f"({int(diff_cls.sum())} edges)")

GAT  | train 1.00 | val 1.00 | test 0.99
mean attention on same-class edges     : 0.145 (562 edges)
mean attention on different-class edges: 0.1335 (120 edges)


### 10. Over-smoothing: why GNNs are usually shallow

**What this cell does (plain language):** we apply the propagation matrix `A_hat` repeatedly (no learning at all) and measure how different the node representations still are from each other. Each application averages a node with its neighbours, so after many hops every node in a connected component converges to nearly the same vector and class information is washed out. We also measure the accuracy of a simple nearest-class-mean classifier on the propagated features, which first rises (noise averaged away) and then eventually falls as nodes become indistinguishable.

In [10]:
def nearest_mean_acc(h):
    # classify with class means computed on the 15 training nodes only
    mus = torch.stack([h[train_mask & (y == c)].mean(0) for c in range(n_classes)])
    pred = torch.cdist(h, mus).argmin(1)
    return (pred[test_mask] == y[test_mask]).float().mean().item()

h = X.clone()
print("hops | std of features across nodes | nearest-mean test acc")
for k in range(0, 41):
    if k in (0, 1, 2, 4, 8, 16, 40):
        print(f"{k:4d} | {h.std(0).mean().item():28.4f} | {nearest_mean_acc(h):.2f}")
    h = A_hat @ h

hops | std of features across nodes | nearest-mean test acc
   0 |                       1.0911 | 0.75
   1 |                       0.5328 | 0.97
   2 |                       0.3895 | 0.99
   4 |                       0.2752 | 0.93
   8 |                       0.1835 | 0.80
  16 |                       0.1309 | 0.53
  40 |                       0.1222 | 0.40


### 11. Permutation equivariance: node order must not matter

**What this cell does (plain language):** a GNN should not care how nodes are numbered. We relabel the nodes with a random permutation, run the same trained GCN on the permuted graph, and check that the outputs are the original outputs with rows reordered in exactly the same way. This property is the reason GNNs are the right tool for graphs; an MLP on a flattened adjacency matrix would not have it.

In [11]:
torch.manual_seed(1)
perm = torch.randperm(N)                     # new node i is old node perm[i]
X_p = X[perm]
A_p = A[perm][:, perm]
gcn.eval()
with torch.no_grad():
    out_orig = gcn(X, A_hat)
    out_perm = gcn(X_p, normalize_adj(A_p))
print("GCN(permuted graph) == permuted GCN(graph):", torch.allclose(out_perm, out_orig[perm], atol=1e-5))

GCN(permuted graph) == permuted GCN(graph): True


## Common bugs

- **Forgetting self-loops**
  A node's own features are dropped (it only sees neighbours); isolated nodes become all-zero.
  Fix: use `A + I` before normalising.
- **Dividing by zero degree**
  Isolated nodes have degree 0, giving `inf`/`nan` in `D^-1/2`. Fix: add self-loops or `clamp(min=1)`.
- **Normalising the wrong axis / not symmetrically**
  Using `A / A.sum(0)` vs `A / A.sum(1)` changes meaning (column vs row normalisation) and breaks symmetry. Fix: be explicit; `D^-1/2 A D^-1/2` for GCN, `D^-1 A` for mean aggregation.
- **Edge direction mixed up (`src`/`dst`)**
  Messages travel the wrong way on directed graphs. Fix: be consistent: `edge_index[0]` is the source being gathered, `edge_index[1]` is the destination being scattered to.
- **Computing the loss on all nodes when only some are labelled** (or leaking test labels into training)
  Fix: index logits with the train mask; evaluate on masks that never touched training.
- **Softmax over the wrong dimension in attention**
  Softmax must be over each node's incoming edges, not over all edges or all nodes. Fix: per-destination softmax (scatter max / sum) or masked dense softmax with `-inf` on non-edges.
- **Masked dense attention with `0` instead of `-inf`**
  Non-edges still receive weight `exp(0)`. Fix: fill with `-inf` (or a large negative) before softmax.
- **Too many layers**
  Over-smoothing: all node representations collapse. Fix: 2-3 layers, residual connections, or other tricks.
- **Dense `N x N` adjacency on large graphs**
  Out of memory. Fix: sparse tensors or edge-list gather/scatter.
- **`torch.sparse` tensors that are not coalesced**
  Duplicate indices can give wrong results in some ops. Fix: call `.coalesce()`.

## Exercises

Attempt each one in the empty cell right after it, then compare with the solution below.

**Exercise 1 - Row-normalised adjacency.**
Compute `D^-1 (A + I)` (random-walk normalisation) and verify every row sums to 1. How does one propagation step differ from the symmetric one?

In [12]:
# Your attempt for Exercise 1 here

**Solution 1**

In [13]:
# Solution 1
A_t = A + torch.eye(N)
A_rw = A_t / A_t.sum(1, keepdim=True)
print("rows sum to 1:", torch.allclose(A_rw.sum(1), torch.ones(N)))
print("symmetric?", torch.allclose(A_rw, A_rw.T), "(no: it is an average over neighbours, so it is not symmetric when degrees differ)")

rows sum to 1: True
symmetric? False (no: it is an average over neighbours, so it is not symmetric when degrees differ)


**Exercise 2 - Count the receptive field.**
For node 0, count how many nodes are reachable within 1 and 2 hops (including itself) using powers of `A + I`.

In [14]:
# Your attempt for Exercise 2 here

**Solution 2**

In [15]:
# Solution 2
R = A + torch.eye(N)
r1 = (R[0] > 0).sum().item()
r2 = ((R @ R)[0] > 0).sum().item()
print("nodes within 1 hop:", r1, "| within 2 hops:", r2)

nodes within 1 hop: 7 | within 2 hops: 37


**Exercise 3 - GCN with 1 layer vs 2 layers vs 3 layers.**
Build a GCN with a configurable depth by stacking `DenseGCNLayer`s and compare test accuracy for depths 1, 2, 3 (use `fit`). Print the results.

In [16]:
# Your attempt for Exercise 3 here

**Solution 3**

In [17]:
# Solution 3
class DeepGCN(nn.Module):
    def __init__(self, depth, p=0.5):
        super().__init__()
        dims = [F_in] + [16] * (depth - 1) + [n_classes]
        self.layers = nn.ModuleList([DenseGCNLayer(dims[i], dims[i + 1]) for i in range(depth)])
        self.p = p
    def forward(self, x, A_hat):
        for i, l in enumerate(self.layers):
            x = F.dropout(x, self.p, self.training)
            x = l(x, A_hat)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return x
for depth in [1, 2, 3]:
    torch.manual_seed(0)
    m = DeepGCN(depth)
    print("depth", depth, "-> (train, val, test):", [round(v, 2) for v in fit(m, lambda: m(X, A_hat), verbose=False)])

depth 1 -> (train, val, test): [1.0, 0.97, 0.95]


depth 2 -> (train, val, test): [1.0, 1.0, 0.93]


depth 3 -> (train, val, test): [1.0, 1.0, 0.96]


**Exercise 4 - Graph-level readout.**
Turn node embeddings into one graph vector with mean pooling and sum pooling, using the output of the first trained GCN layer (`gcn.l1`). Show the shapes.

In [18]:
# Your attempt for Exercise 4 here

**Solution 4**

In [19]:
# Solution 4
with torch.no_grad():
    H = F.relu(gcn.l1(X, A_hat))
print("node embeddings:", tuple(H.shape), "| mean readout:", tuple(H.mean(0).shape), "| sum readout:", tuple(H.sum(0).shape))

node embeddings: (120, 16) | mean readout: (16,) | sum readout: (16,)


**Exercise 5 - Dense masked attention.**
Re-implement the attention weights densely: compute the N x N score matrix, fill non-edges (including after adding self-loops) with `-inf`, softmax over rows. Check that the weights for node 0 match the edge-list `GATLayer` for the same parameters (use `gat_layer`, the untrained single layer).

In [20]:
# Your attempt for Exercise 5 here

**Solution 5**

In [21]:
# Solution 5
with torch.no_grad():
    h = gat_layer.W(X)
    e = F.leaky_relu((h @ gat_layer.a_dst)[:, None] + (h @ gat_layer.a_src)[None, :], 0.2)   # e[i, j]: dst i attends to src j
    mask = (A + torch.eye(N)) > 0
    alpha_dense = torch.softmax(e.masked_fill(~mask, float("-inf")), dim=1)
    out_dense = alpha_dense @ h
    out_edge = gat_layer(X, edge_index)
print("dense masked attention == edge-list attention:", torch.allclose(out_dense, out_edge, atol=1e-5))

dense masked attention == edge-list attention: True


**Exercise 6 - Label-uninformative graph.**
Shuffle the edges so the graph carries no label information (random permutation of destination nodes) and train the GCN. Does it still beat the MLP? Explain what you observe.

In [22]:
# Your attempt for Exercise 6 here

**Solution 6**

In [23]:
# Solution 6
torch.manual_seed(0)
rand_perm = torch.randperm(N)
A_bad = A[rand_perm][:, rand_perm]            # same structure, but unrelated to the labels
A_hat_bad = normalize_adj(A_bad)
m = GCN(F_in, 16, n_classes)
print("GCN on label-uninformative graph (train, val, test):", [round(v, 2) for v in fit(m, lambda: m(X, A_hat_bad), verbose=False)])
print("MLP test:", round(te_m, 2), "-> the graph helps only when it is homophilous.")

GCN on label-uninformative graph (train, val, test): [0.67, 0.7, 0.39]
MLP test: 0.71 -> the graph helps only when it is homophilous.
